# Introduction

This notebook addresses our mission to analyze and predict student reading performance based on demographic, academic, and school factors from the PISA 2009 dataset. We will train several machine learning models to identify key variables that affect student outcomes and select the best model to deploy in our API.

# Load Dataset

We load the PISA 2009 train and test datasets.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

train_path = 'data/pisa2009train.csv'
test_path = 'data/pisa2009test.csv'

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)

# Dataset Overview

We display the first 5 rows, dataset info, and descriptive statistics.

In [ ]:
df_train.head()

In [ ]:
df_train.info()

In [ ]:
df_train.describe()

# Missing Values

Let's check the number of missing values in both datasets.

In [ ]:
df_train.isnull().sum()

In [ ]:
df_test.isnull().sum()

# Data Visualization

We visualize the correlation heatmap and distributions of reading scores.

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(10, 8))
numeric_df = df_train.select_dtypes(include=[np.number])
corr = numeric_df.corr()
sns.heatmap(corr, annot=False, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap of PISA 2009 Dataset')
plt.tight_layout()
plt.show()

In [ ]:
# Reading Score Distribution by Grade
plt.figure(figsize=(8, 6))
sns.boxplot(x='grade', y='readingScore', data=df_train)
plt.title('Distribution of Reading Scores by Grade')
plt.show()

# Feature Engineering and Selection

We select the most significant features for predicting the reading score:
1. `grade`
2. `male`
3. `raceeth`
4. `expectBachelors`
5. `read30MinsADay`
6. `minutesPerWeekEnglish`
7. `studentsInEnglish`
8. `schoolSize`

We drop the rest of the columns to keep our API and mobile app interfaces clean and lightweight.

In [ ]:
features = ['grade', 'male', 'raceeth', 'expectBachelors', 'read30MinsADay', 
            'minutesPerWeekEnglish', 'studentsInEnglish', 'schoolSize']
target = 'readingScore'

X_train = df_train[features]
y_train = df_train[target]
X_test = df_test[features]
y_test = df_test[target]

# Data Preprocessing

We define a ColumnTransformer to handle:
- Median imputation and standard scaling for continuous features.
- Most frequent imputation and one-hot encoding for the categorical feature `raceeth`.
- Most frequent imputation for binary features.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features = ['minutesPerWeekEnglish', 'studentsInEnglish', 'schoolSize']
categorical_features = ['raceeth']
binary_features = ['grade', 'male', 'expectBachelors', 'read30MinsADay']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

binary_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
        ('bin', binary_transformer, binary_features)
    ])

# Fit and transform
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

# Model Training & Comparison

We train and compare four models using Scikit-Learn:
1. Ordinary Least Squares Linear Regression (`LinearRegression`)
2. Stochastic Gradient Descent Regression (`SGDRegressor`)
3. Decision Tree Regressor (`DecisionTreeRegressor`)
4. Random Forest Regressor (`RandomForestRegressor`)

In [ ]:
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, mean_squared_error

# 1. OLS
ols = LinearRegression()
ols.fit(X_train_preprocessed, y_train)
ols_train_rmse = root_mean_squared_error(y_train, ols.predict(X_train_preprocessed))
ols_test_rmse = root_mean_squared_error(y_test, ols.predict(X_test_preprocessed))

# 2. SGD
epochs = 150
sgd = SGDRegressor(max_iter=1, tol=None, warm_start=True, random_state=42, learning_rate='constant', eta0=0.01)
sgd_train_losses = []
sgd_test_losses = []

for epoch in range(epochs):
    indices = np.random.permutation(len(X_train_preprocessed))
    sgd.partial_fit(X_train_preprocessed[indices], y_train.iloc[indices])
    
    y_pred_train = sgd.predict(X_train_preprocessed)
    y_pred_test = sgd.predict(X_test_preprocessed)
    
    sgd_train_losses.append(mean_squared_error(y_train, y_pred_train))
    sgd_test_losses.append(mean_squared_error(y_test, y_pred_test))

sgd_train_rmse = np.sqrt(sgd_train_losses[-1])
sgd_test_rmse = np.sqrt(sgd_test_losses[-1])

# 3. Decision Tree
dt = DecisionTreeRegressor(max_depth=5, random_state=42)
dt.fit(X_train_preprocessed, y_train)
dt_train_rmse = root_mean_squared_error(y_train, dt.predict(X_train_preprocessed))
dt_test_rmse = root_mean_squared_error(y_test, dt.predict(X_test_preprocessed))

# 4. Random Forest
rf = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42)
rf.fit(X_train_preprocessed, y_train)
rf_train_rmse = root_mean_squared_error(y_train, rf.predict(X_train_preprocessed))
rf_test_rmse = root_mean_squared_error(y_test, rf.predict(X_test_preprocessed))

print(f"OLS - Train RMSE: {ols_train_rmse:.4f}, Test RMSE: {ols_test_rmse:.4f}")
print(f"SGD - Train RMSE: {sgd_train_rmse:.4f}, Test RMSE: {sgd_test_rmse:.4f}")
print(f"Decision Tree - Train RMSE: {dt_train_rmse:.4f}, Test RMSE: {dt_test_rmse:.4f}")
print(f"Random Forest - Train RMSE: {rf_train_rmse:.4f}, Test RMSE: {rf_test_rmse:.4f}")

# Plot SGD Loss Curve

We plot the training and test loss curve for the SGD Regressor over epochs.

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(np.sqrt(sgd_train_losses), label='Train Loss (RMSE)')
plt.plot(np.sqrt(sgd_test_losses), label='Test Loss (RMSE)')
plt.xlabel('Epochs')
plt.ylabel('RMSE')
plt.title('SGD Regressor Loss Curve over Epochs')
plt.legend()
plt.grid(True)
plt.show()

# Scatter Plot of Best Fit

We plot the Actual vs Predicted reading score for the best performing model (Random Forest), and a scatter plot showing grade vs readingScore with OLS regression line.

In [ ]:
# Actual vs Predicted
plt.figure(figsize=(8, 6))
plt.scatter(y_test, rf.predict(X_test_preprocessed), alpha=0.3, color='blue', label='Predictions')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Fit')
plt.xlabel('Actual Reading Score')
plt.ylabel('Predicted Reading Score')
plt.title('Actual vs. Predicted Reading Scores (Random Forest)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Linear fit of Grade vs Reading Score
plt.figure(figsize=(8, 6))
sns.scatterplot(x=df_train['grade'], y=df_train['readingScore'], alpha=0.1, color='blue')
m, c = np.polyfit(df_train['grade'], df_train['readingScore'], 1)
x_vals = np.array([8, 12])
plt.plot(x_vals, m*x_vals + c, color='red', lw=2, label=f'Linear Fit: y = {m:.2f}x + {c:.2f}')
plt.xlabel('Grade')
plt.ylabel('Reading Score')
plt.title('Linear Fit of Reading Score by Grade')
plt.legend()
plt.grid(True)
plt.show()

# Make Prediction on One Test Row

We select the first row of the test dataset, preprocess it, and predict its reading score.

In [ ]:
# Predict on first row of test set
test_row = X_test.iloc[[0]]
print("Test Row Input:")
print(test_row)

test_row_preprocessed = preprocessor.transform(test_row)
actual_score = y_test.iloc[0]
predicted_score = rf.predict(test_row_preprocessed)[0]

print(f"\nActual Score: {actual_score}")
print(f"Predicted Score: {predicted_score:.4f}")
print(f"Absolute Error: {abs(actual_score - predicted_score):.4f}")

# Save Best Model

We save the best model and preprocessing pipeline.

In [ ]:
# Save pipeline and best model
joblib.dump(preprocessor, '../API/preprocessor.joblib')
joblib.dump(rf, '../API/best_model.joblib')
print("Best model and preprocessor saved successfully.")